# ReSA under class imbalance — experiments notebook

This notebook runs every experiment of the study *"ReSA self-supervised learning on long-tailed data"* on Google Colab, with the code and the datasets stored on Google Drive.

**What the study asks.** ReSA (Weng et al., ICML 2025) learns image representations without labels. Its published results use balanced datasets. Here ReSA is pretrained on **long-tailed** versions of CIFAR-10 and CIFAR-100 (imbalance factors 20, 50 and 100: the largest class has 20×, 50× or 100× more images than the smallest), and the frozen encoder is then evaluated in three ways that separate *what the encoder learned* from *how the classifier on top was trained*:

| Probe | Classifier is trained on | What it tells us |
|---|---|---|
| `long_tailed` | the same long-tailed subset used for pretraining | the end-to-end result under imbalance |
| `rebalanced` | the same subset, with every class given equal weight | how much of the loss is classifier bias that re-weighting removes |
| `balanced` | the full balanced training set | the quality of the representation itself |

All three are always tested on the official **balanced** test set.

**How the notebook is organised**

| Section | Cells |
|---|---|
| 0. Setup | mount Drive · **configuration (edit this one)** · environment · datasets |
| 1. Pretraining | one run · **resume after a disconnect** · **all imbalance ratios in one go** · progress |
| 2. Downstream evaluation | **k-NN (k = 5, 10, 20) for one probe** · k-NN for everything · linear probe |
| 3. Transfer | CIFAR-10 encoder → Tiny ImageNet / CIFAR-100 / CIFAR-10 · CIFAR-100 encoder → Tiny ImageNet / CIFAR-10 / CIFAR-100 |
| 4. Results | collect everything into CSV · figures and raw data for one run · compare the three probes |
| Appendix | two-minute smoke test |

**After a Colab disconnect:** run the three Setup cells marked ▶ (mount, configuration, environment), then the *Resume* cell in section 1. Nothing is lost: every epoch is saved to Drive.

The notebook only *launches* experiments. The logic lives in the Python files of the repository (`main.py`, `eval_knn.py`, `eval_linear_cifar.py`, `eval_transfer.py`, `visualise_results.py`, `collect_results.py`); each cell prints the exact command it runs, so any result can be reproduced from a terminal. See `README.md`.

## 0. Setup

In [ ]:
#@title ▶ 0.1 Mount Google Drive
# The code, the datasets and all results live on Drive, so they survive
# when the Colab machine is recycled.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@title ▶ 0.2 Configuration — every setting of the study is here { display-mode: "form" }
# Edit the values, then run this cell. Every other cell reads its settings
# from here, so this is the only place that needs changing.

# ---- Where things are on Google Drive ---------------------------------------
CODE_DIR     = "/content/drive/MyDrive/ReSA_Research/resa_cifar_lt"  #@param {type:"string"}
DATA_ROOT    = "/content/drive/MyDrive/Datasets"                     #@param {type:"string"}
RESULTS_ROOT = "/content/drive/MyDrive/ReSA_Research/results"        #@param {type:"string"}

# ---- The single run that the "one run" cells work on ------------------------
DATASET   = "cifar10"   #@param ["cifar10", "cifar100"]
IMB_RATIO = 100         #@param {type:"integer"}
# IMB_RATIO = largest class size / smallest class size. 1 means balanced.

# ---- The sweeps ("run everything" cells) ------------------------------------
SWEEP_DATASETS = ["cifar10", "cifar100"]
IMB_RATIOS     = [20, 50, 100]          # add 1 to include balanced pretraining
K_VALUES       = [5, 10, 20]            # neighbours for k-NN

# ---- Pretraining -------------------------------------------------------------
PRETRAIN_EPOCHS     = 1000   #@param {type:"integer"}
PRETRAIN_BATCH_SIZE = 256    #@param {type:"integer"}
SEED                = 0      #@param {type:"integer"}
# The seed also fixes WHICH images form the long-tailed subset. Seed 0 gives
# the standard CIFAR-LT subset. Runs with another seed get a "_seed<N>" suffix.

# ---- Linear classifier (linear probe and transfer) ---------------------------
# The defaults are the CIFAR linear evaluation protocol of the ReSA paper
# (Weng et al., ICML 2025, Appendix B.3), inherited from W-MSE and INTL:
# Adam, 500 epochs, learning rate decayed exponentially from 1e-2 to 1e-6,
# weight decay 5e-6, batch size 1000, no augmentation, final epoch reported.
# The features of the frozen encoder are extracted once, so 500 epochs take
# about a minute, not hours.
PROBE_OPTIMIZER  = "adam"   #@param ["adam", "sgd"]
PROBE_EPOCHS     = 500      #@param {type:"integer"}
PROBE_LR         = 0.01     #@param {type:"number"}
PROBE_LR_END     = 1e-6     #@param {type:"number"}
PROBE_WD         = 5e-6     #@param {type:"number"}
PROBE_BATCH_SIZE = 1000     #@param {type:"integer"}
# "sgd" is the earlier scaffold of this repository (momentum 0.9, cosine
# schedule); the values used with it were 100 epochs, lr 10.0, batch 256, no
# weight decay. Use it only to reproduce numbers produced with it, and say in
# the write-up which protocol a number comes from.

# ---- Other -------------------------------------------------------------------
WORKERS   = 2       #@param {type:"integer"}
USE_WANDB = False   #@param {type:"boolean"}
# Weights & Biases logging is off by default. If you switch it on, store your
# API key in Colab's Secrets panel (key icon, name WANDB_API_KEY) -- never
# paste a key into a notebook cell.

# Existing result folders that do not follow the standard naming. Left side:
# the standard name; right side: the folder that already exists on Drive.
RUN_NAMES = {
    "cifar100_balanced": "cifar100",
    # "cifar10_lt20_seed42": "cifar10_lt20_S2",
}

# ------------------------------------------------------------------------------
import os, sys
assert os.path.isfile(os.path.join(CODE_DIR, "experiments.py")), (
    f"The code was not found in {CODE_DIR}. Upload the repository folder to Drive "
    "(or clone it there) and set CODE_DIR to it.")
os.chdir(CODE_DIR)
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
import importlib, experiments as ex
importlib.reload(ex)

cfg = ex.config(
    results_root=RESULTS_ROOT, data_root=DATA_ROOT, seed=SEED,
    pretrain_epochs=PRETRAIN_EPOCHS, pretrain_batch_size=PRETRAIN_BATCH_SIZE,
    probe_optimizer=PROBE_OPTIMIZER, probe_epochs=PROBE_EPOCHS, probe_lr=PROBE_LR,
    probe_lr_end=PROBE_LR_END, probe_wd=PROBE_WD, probe_batch_size=PROBE_BATCH_SIZE,
    workers=WORKERS, use_wandb=USE_WANDB, run_names=RUN_NAMES,
)
os.makedirs(RESULTS_ROOT, exist_ok=True)
os.makedirs(DATA_ROOT, exist_ok=True)

if USE_WANDB:
    from google.colab import userdata
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")

print(f"code      {CODE_DIR}\ndata      {DATA_ROOT}\nresults   {RESULTS_ROOT}")
print(f"single run: {DATASET}, imbalance ratio {IMB_RATIO}  ->  {ex.run_name(cfg, DATASET, IMB_RATIO)}")
print(f"sweeps:     {SWEEP_DATASETS} x imbalance ratios {IMB_RATIOS};  k = {K_VALUES}")
print(f"linear probe: {PROBE_OPTIMIZER}, {PROBE_EPOCHS} epochs, lr {PROBE_LR}"
      + (f" -> {PROBE_LR_END}" if PROBE_OPTIMIZER == "adam" else "")
      + f", weight decay {PROBE_WD}, batch {PROBE_BATCH_SIZE}")

In [ ]:
#@title ▶ 0.3 Environment
# Colab already provides PyTorch, torchvision, NumPy, scikit-learn, pandas and
# matplotlib; this only adds what may be missing and reports the versions
# (quote them in the dissertation).
!pip install -q tqdm scikit-learn pandas matplotlib
if USE_WANDB:
    !pip install -q wandb

import torch, torchvision, numpy, sklearn
print(f"torch {torch.__version__} | torchvision {torchvision.__version__} | numpy {numpy.__version__} | scikit-learn {sklearn.__version__}")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU. Runtime > Change runtime type > GPU. (Pretraining on CPU is not practical.)")

In [ ]:
#@title 0.4 Datasets (run once)
# Expected content of DATA_ROOT:
#   cifar-10-batches-py/      CIFAR-10   (downloaded here if missing)
#   cifar-100-python/         CIFAR-100  (downloaded here if missing)
#   tiny-imagenet-200.zip     Tiny ImageNet, only for the transfer section
#                             (downloaded here if missing, ~248 MB)
# Tiny ImageNet is 120,000 small JPEG files. Reading them one at a time from
# Drive is very slow, so the .zip is copied to the local disk, read once,
# resized to 32x32 -- the resolution the encoders were trained at -- and
# stored as two .npz files.
#
# Every step prints what it is doing and a progress line about every ten
# seconds. Typical time: 1-2 minutes if the files are already on Drive,
# 5-15 minutes on the very first run. Downloads go to the local disk first
# and are then copied to Drive. Running the cell again skips what is done.
import os, shutil, time, importlib
import torchvision
import src.tiny_imagenet as tiny_imagenet
importlib.reload(tiny_imagenet)   # pick up the current file on Drive

t0 = time.time()
LOCAL_DL = "/content/_dataset_downloads"
for name, cls, folder in (("CIFAR-10", torchvision.datasets.CIFAR10, "cifar-10-batches-py"),
                          ("CIFAR-100", torchvision.datasets.CIFAR100, "cifar-100-python")):
    if not os.path.isdir(os.path.join(DATA_ROOT, folder)):
        print(f"{name}: not on Drive yet, downloading to the local disk ...", flush=True)
        cls(LOCAL_DL, train=True, download=True)
        print(f"{name}: copying to {DATA_ROOT} ...", flush=True)
        tmp = os.path.join(DATA_ROOT, folder + ".incomplete")
        shutil.rmtree(tmp, ignore_errors=True)
        shutil.copytree(os.path.join(LOCAL_DL, folder), tmp)
        os.rename(tmp, os.path.join(DATA_ROOT, folder))
    print(f"{name}: reading from Drive ...", flush=True)
    for train in (True, False):
        d = cls(DATA_ROOT, train=train, download=False)
    print(f"{name}: ready ({len(d.classes)} classes)  [{time.time() - t0:.0f} s]", flush=True)

PREPARE_TINY_IMAGENET = True  #@param {type:"boolean"}
# Only the transfer section (3) needs Tiny ImageNet; untick to skip it for now.
if PREPARE_TINY_IMAGENET:
    for split in ("train", "val"):
        x, y, classes = tiny_imagenet.load_tiny_imagenet(DATA_ROOT, split, img_size=32)
        print(f"Tiny ImageNet {split}: {x.shape[0]:,} images, {len(classes)} classes, "
              f"{x.shape[1]}x{x.shape[2]} px  [{time.time() - t0:.0f} s]", flush=True)
print("All datasets are ready.")

## 1. Pretraining

ReSA is trained without labels on the chosen dataset. With `IMB_RATIO > 1` the training set is the long-tailed subset (class *i* keeps `n_max · ratio^(−i/(C−1))` images, the standard CIFAR-LT protocol); with `IMB_RATIO = 1` it is the full balanced set.

Each run writes to `RESULTS_ROOT/<dataset>_lt<ratio>/`:

* `checkpoint.pth.tar` — saved after **every epoch** (the momentum encoder inside it is what all evaluations use)
* `status.json` — epochs done / planned
* `args.json` — every setting of the run
* `out.log` — the training log

A full 1000-epoch run takes many hours and will usually outlive a Colab session. That is expected: use the *Resume* cell.

In [ ]:
#@title 1.1 Pretrain one run (DATASET and IMB_RATIO from the configuration cell)
# Starts the run, or continues it if a checkpoint already exists.
ex.pretrain(cfg, DATASET, IMB_RATIO)

In [ ]:
#@title 1.2 Resume after a Colab disconnect
# Run cells 0.1, 0.2 and 0.3 first, then this cell.
# It lists every pretraining run of the study and continues the interrupted
# one(s) from the last saved epoch -- same learning-rate schedule, same
# long-tailed subset. Finished runs are left alone; runs that were never
# started are NOT started here (use 1.1 or 1.3 for that).
print("Before:")
ex.pretraining_status(cfg, SWEEP_DATASETS, sorted(set(IMB_RATIOS + [IMB_RATIO, 1])))
print()
resumed = 0
for dataset in SWEEP_DATASETS:
    for ratio in sorted(set(IMB_RATIOS + [IMB_RATIO, 1])):
        st = ex.read_status(cfg, dataset, ratio)
        if st is not None and st.get("done") is not True:
            print(f"\n=== resuming {ex.run_name(cfg, dataset, ratio)} ===")
            ex.pretrain(cfg, dataset, ratio)
            resumed += 1
print("\nNo interrupted run found." if resumed == 0 else f"\n{resumed} run(s) resumed and completed.")

In [ ]:
#@title 1.3 Pretrain ALL imbalance ratios in one go
# Runs SWEEP_DATASETS x IMB_RATIOS one after the other. Finished runs are
# skipped and interrupted runs are continued, so after a disconnect simply run
# this cell again (after 0.1-0.3) and the sweep carries on where it stopped.
for dataset in SWEEP_DATASETS:
    for ratio in IMB_RATIOS:
        print(f"\n================ {dataset} | imbalance ratio {ratio} ================")
        ex.pretrain(cfg, dataset, ratio)
print("\nAll pretraining runs are complete.")

In [ ]:
#@title 1.4 Progress of all pretraining runs
_ = ex.pretraining_status(cfg, SWEEP_DATASETS, sorted(set(IMB_RATIOS + [1])))

## 2. Downstream evaluation on the pretraining dataset

The encoder is frozen. Two kinds of classifier are placed on top of its 512-dimensional features:

* **k-NN** — no training at all; a test image gets the majority label of its *k* nearest training images (Euclidean distance on the raw features, the ReSA CIFAR protocol). It measures how well classes are separated in feature space.
* **Linear probe** — one linear layer trained on the frozen features with the paper's CIFAR protocol (Adam, `PROBE_EPOCHS` = 500 epochs, learning rate `PROBE_LR` = 1e-2 decayed exponentially to `PROBE_LR_END` = 1e-6, weight decay 5e-6, batch 1000, no augmentation, final epoch reported). Set `PROBE_OPTIMIZER = "sgd"` in the configuration cell for the earlier scaffold of this repository.

Both exist in the three variants from the table at the top. For k-NN, `rebalanced` means each neighbour's vote is divided by the size of its class in the long-tailed subset, so a head class cannot win merely by being common.

Results are written to `RESULTS_ROOT/<run>_knn_<probe>/metrics/` and `RESULTS_ROOT/<run>_probe_<probe>/metrics/`.

In [ ]:
#@title 2.1 k-NN: choose the probe, the dataset and the k values { display-mode: "form" }
KNN_PROBE     = "long_tailed"   #@param ["long_tailed", "rebalanced", "balanced"]
KNN_DATASET   = "cifar10"       #@param ["cifar10", "cifar100"]
KNN_IMB_RATIO = 100             #@param {type:"integer"}
KNN_K         = "5, 10, 20"     #@param {type:"string"}
# KNN_IMB_RATIO selects which pretrained encoder is evaluated (the one
# pretrained at that imbalance ratio) and defines the long-tailed subset.

import pandas as pd
k_values = [int(k) for k in KNN_K.replace(",", " ").split()]
ex.knn(cfg, KNN_DATASET, KNN_IMB_RATIO, KNN_PROBE, k_values)

# The CSV with one row per k: accuracy, macro precision / recall / F1 / AUC,
# balanced accuracy, head / medium / tail accuracy, worst-class recall, ...
knn_csv = os.path.join(ex.eval_dir(cfg, KNN_DATASET, KNN_IMB_RATIO, "knn", KNN_PROBE), "metrics", "knn_summary.csv")
print("\nSaved:", knn_csv)
pd.read_csv(knn_csv)[["dataset", "probe", "imb_ratio", "k", "acc1", "macro_precision", "macro_recall", "macro_f1",
                      "macro_auc", "balanced_accuracy", "head_acc", "mid_acc", "tail_acc", "min_class_recall"]]

In [ ]:
#@title 2.2 k-NN for everything: both datasets x all imbalance ratios x three probes x all k
# 2 datasets x 3 ratios x 3 probes = 18 evaluations; each takes about a minute
# on a GPU. Runs whose pretraining checkpoint does not exist yet are skipped.
import pandas as pd
for dataset in SWEEP_DATASETS:
    for ratio in IMB_RATIOS:
        for probe in ex.PROBES:
            print(f"\n================ k-NN | {dataset} | ratio {ratio} | {probe} ================")
            try:
                ex.knn(cfg, dataset, ratio, probe, K_VALUES)
            except FileNotFoundError as e:
                print("[skipped]", e)

# one table with every k-NN result so far, also saved as a CSV
frames = []
for dataset in SWEEP_DATASETS:
    for ratio in IMB_RATIOS:
        for probe in ex.PROBES:
            p = os.path.join(ex.eval_dir(cfg, dataset, ratio, "knn", probe), "metrics", "knn_summary.csv")
            if os.path.isfile(p):
                frames.append(pd.read_csv(p))
if frames:
    knn_all = pd.concat(frames, ignore_index=True)
    out = os.path.join(RESULTS_ROOT, "knn_all_results.csv")
    knn_all.to_csv(out, index=False)
    print("\nSaved:", out)
    display(knn_all.pivot_table(index=["dataset", "imb_ratio", "probe"], columns="k", values="acc1").round(2))

In [ ]:
#@title 2.3 Linear probe: choose the probe and the dataset { display-mode: "form" }
LIN_PROBE     = "long_tailed"   #@param ["long_tailed", "rebalanced", "balanced"]
LIN_DATASET   = "cifar10"       #@param ["cifar10", "cifar100"]
LIN_IMB_RATIO = 100             #@param {type:"integer"}
LIN_OVERWRITE = False           #@param {type:"boolean"}
# An interrupted probe continues where it stopped; a finished one is not
# repeated unless LIN_OVERWRITE is ticked.

import pandas as pd
ex.linear_probe(cfg, LIN_DATASET, LIN_IMB_RATIO, LIN_PROBE, overwrite=LIN_OVERWRITE)
lin_csv = os.path.join(ex.eval_dir(cfg, LIN_DATASET, LIN_IMB_RATIO, "probe", LIN_PROBE), "metrics", "summary_metrics_final.csv")
print("\nSaved:", lin_csv)
pd.read_csv(lin_csv)

In [ ]:
#@title 2.4 Linear probe for everything: both datasets x all imbalance ratios x three probes
# 18 probes of PROBE_EPOCHS epochs each -- allow a few hours on a GPU.
# Safe to re-run after a disconnect: finished probes are skipped.
for dataset in SWEEP_DATASETS:
    for ratio in IMB_RATIOS:
        for probe in ex.PROBES:
            print(f"\n================ linear probe | {dataset} | ratio {ratio} | {probe} ================")
            try:
                ex.linear_probe(cfg, dataset, ratio, probe)
            except FileNotFoundError as e:
                print("[skipped]", e)

## 3. Transfer to other datasets

How general are the features? The encoder pretrained on one dataset (the *source*) is frozen and used, unchanged, on another dataset (the *target*): a linear classifier and k-NN classifiers (k = 5, 10, 20) are fitted on the **full training set of the target** and tested on the target's test set.

* CIFAR-10 encoder → Tiny ImageNet, CIFAR-100, CIFAR-10
* CIFAR-100 encoder → Tiny ImageNet, CIFAR-10, CIFAR-100

(source = target is the in-domain reference the transfer numbers are compared with.)

Tiny ImageNet images (64×64) are resized to 32×32, the resolution the encoders were trained at; its 10,000 labelled validation images serve as the test set. Images are normalised with the *source* dataset's statistics, because that is what the frozen encoder expects.

`TRANSFER_SOURCE_RATIOS` chooses which pretrained encoders are used as sources — the one pretrained on balanced data (1) and/or the ones pretrained at each imbalance ratio. Using all of them shows how pretraining imbalance affects transferability. Sources that have not been pretrained yet are skipped. `TRANSFER_TARGETS` chooses the targets; the tables show every finished target.

Results: `RESULTS_ROOT/transfer/<source run>__to__<target>/metrics/transfer_summary.csv`

In [ ]:
#@title 3.1 Which source encoders to use { display-mode: "form" }
TRANSFER_SOURCE_RATIOS = "1, 20, 50, 100"   #@param {type:"string"}
TRANSFER_TARGETS       = "tiny_imagenet, cifar100, cifar10"   #@param {type:"string"}
ALL_TARGETS = ["tiny_imagenet", "cifar100", "cifar10"]
transfer_targets = TRANSFER_TARGETS.replace(",", " ").split()
assert set(transfer_targets) <= set(ex.TARGETS), f"unknown target in {transfer_targets}; choose from {ex.TARGETS}"
transfer_ratios = [int(r) for r in TRANSFER_SOURCE_RATIOS.replace(",", " ").split()]

def run_transfer(source_dataset):
    for ratio in transfer_ratios:
        for target in transfer_targets:
            print(f"\n================ {ex.run_name(cfg, source_dataset, ratio)}  ->  {target} ================")
            try:
                ex.transfer(cfg, source_dataset, ratio, target, K_VALUES)
            except FileNotFoundError as e:
                print("[skipped]", e)

def show_transfer(source_dataset):
    import pandas as pd
    frames = []
    for ratio in transfer_ratios:
        for target in ALL_TARGETS:   # every finished target, not only the ones just run
            p = os.path.join(ex.transfer_dir(cfg, source_dataset, ratio, target), "metrics", "transfer_summary.csv")
            if os.path.isfile(p):
                d = pd.read_csv(p)
                d.insert(0, "source_run", ex.run_name(cfg, source_dataset, ratio))
                frames.append(d)
    if not frames:
        print("no transfer results yet")
        return
    table = pd.concat(frames, ignore_index=True)
    out = os.path.join(RESULTS_ROOT, f"transfer_from_{source_dataset}.csv")
    table.to_csv(out, index=False)
    print("Saved:", out)
    display(table.pivot_table(index=["source_run", "target"], columns="classifier", values="acc1", sort=False).round(2))
print("sources:", transfer_ratios, "| targets:", transfer_targets)

In [ ]:
#@title 3.2 CIFAR-10 pretrained encoder -> Tiny ImageNet, CIFAR-100, CIFAR-10 (linear + k-NN)
run_transfer("cifar10")
show_transfer("cifar10")

In [ ]:
#@title 3.3 CIFAR-100 pretrained encoder -> Tiny ImageNet, CIFAR-10, CIFAR-100 (linear + k-NN)
run_transfer("cifar100")
show_transfer("cifar100")

## 4. Results: tables, figures and raw data

**Units.** `acc1` and `acc5` are percentages (0–100). Every other metric (precision, recall, F1, AUC, balanced accuracy, head / medium / tail accuracy) is a fraction between 0 and 1.

**Class groups.** *Head / medium / tail* = the 30 % most frequent classes / the middle 40 % / the 30 % least frequent, by their size in the long-tailed subset (3 / 4 / 3 classes for CIFAR-10). *Many / medium / few-shot* = more than 100 / 20 to 100 / fewer than 20 training images. The groups always refer to the long-tailed subset, also for the balanced probe, so the three probes are compared on the same classes.

In [ ]:
#@title 4.1 Collect every result into CSV files
# Walks RESULTS_ROOT and writes
#   all_results.csv         one row per (run, classifier): the summary table
#   all_per_class.csv       one row per (run, classifier, class): the raw data
#   pretraining_status.csv  progress of the pretraining runs
# Runs made with earlier versions of the code are included as well.
import importlib, pandas as pd, collect_results
importlib.reload(collect_results)
collect_results.collect(RESULTS_ROOT)
results = pd.read_csv(os.path.join(RESULTS_ROOT, "all_results.csv"))
cols = [c for c in ["run", "kind", "dataset", "probe", "imb_ratio", "classifier", "acc1", "macro_precision",
                    "macro_recall", "macro_f1", "macro_auc", "balanced_accuracy", "head_acc", "mid_acc", "tail_acc"]
        if c in results.columns]
results[cols]

In [ ]:
#@title 4.2 Figures and raw-data CSV for one evaluation { display-mode: "form" }
VIS_KIND      = "knn"           #@param ["knn", "linear_probe", "transfer"]
VIS_DATASET   = "cifar10"       #@param ["cifar10", "cifar100"]
VIS_IMB_RATIO = 100             #@param {type:"integer"}
VIS_PROBE     = "long_tailed"   #@param ["long_tailed", "rebalanced", "balanced"]
VIS_K         = 10              #@param {type:"integer"}
VIS_TARGET    = "tiny_imagenet" #@param ["tiny_imagenet", "cifar100", "cifar10"]
VIS_TRANSFER_CLASSIFIER = "linear"  #@param ["linear", "knn"]
# knn:          VIS_DATASET, VIS_IMB_RATIO, VIS_PROBE, VIS_K
# linear_probe: VIS_DATASET, VIS_IMB_RATIO, VIS_PROBE
# transfer:     VIS_DATASET (= source), VIS_IMB_RATIO (of the source), VIS_TARGET,
#               VIS_TRANSFER_CLASSIFIER (and VIS_K if it is knn)
#
# Written to <run>/metrics/figures/:
#   precision_recall_<tag>.png         precision and recall per class, one graph
#   f1_<tag>.png                       F1 per class
#   auc_<tag>.png                      one-vs-rest AUC per class
#   confusion_pairs_<tag>.png          the most frequent confusions
#   confusion_matrix_<tag>.png         counts and row-normalised
#   prediction_distribution_<tag>.png  images assigned to each class vs images truly in it
#   raw_per_class_<tag>.csv, raw_confusion_pairs_<tag>.csv,
#   raw_confusion_matrix_<tag>.csv, raw_summary_<tag>.csv   <- the raw data behind the figures
import importlib, visualise_results as vis
from IPython.display import Image, display
importlib.reload(vis)

if VIS_KIND == "knn":
    run_folder, tag = ex.eval_dir(cfg, VIS_DATASET, VIS_IMB_RATIO, "knn", VIS_PROBE), f"k{VIS_K}"
elif VIS_KIND == "linear_probe":
    run_folder, tag = ex.eval_dir(cfg, VIS_DATASET, VIS_IMB_RATIO, "probe", VIS_PROBE), "final"
else:
    run_folder = ex.transfer_dir(cfg, VIS_DATASET, VIS_IMB_RATIO, VIS_TARGET)
    tag = "linear" if VIS_TRANSFER_CLASSIFIER == "linear" else f"k{VIS_K}"

fig_dir = vis.visualise(os.path.join(run_folder, "metrics"), tag)
for name in ("precision_recall", "f1", "auc", "confusion_pairs", "confusion_matrix", "prediction_distribution"):
    path = os.path.join(fig_dir, f"{name}_{tag}.png")
    if os.path.isfile(path):
        display(Image(path, width=950))

In [ ]:
#@title 4.3 Figures and raw data for EVERY finished evaluation
# Runs the visualiser on each metrics folder under RESULTS_ROOT (all tags).
import importlib, visualise_results as vis
importlib.reload(vis)
done = 0
for dirpath, dirnames, filenames in os.walk(RESULTS_ROOT):
    if os.path.basename(dirpath) == "metrics":
        for tag in vis.available_tags(dirpath):
            if tag == "best":      # 'best' is selected on the test set; report 'final'
                continue
            try:
                vis.visualise(dirpath, tag)
                done += 1
            except Exception as e:
                print(f"[skipped] {dirpath} [{tag}]: {e}")
print(f"\n{done} evaluations visualised.")

In [ ]:
#@title 4.4 Compare the three probes in one graph (recall per class) { display-mode: "form" }
CMP_KIND      = "knn"       #@param ["knn", "linear_probe"]
CMP_DATASET   = "cifar10"   #@param ["cifar10", "cifar100"]
CMP_IMB_RATIO = 100         #@param {type:"integer"}
CMP_K         = 10          #@param {type:"integer"}
import importlib, visualise_results as vis
from IPython.display import Image, display
importlib.reload(vis)
kind, tag = ("knn", f"k{CMP_K}") if CMP_KIND == "knn" else ("probe", "final")
folders = [os.path.join(ex.eval_dir(cfg, CMP_DATASET, CMP_IMB_RATIO, kind, p), "metrics") for p in ex.PROBES]
out_dir = os.path.join(RESULTS_ROOT, "figures")
name = f"compare_{ex.run_name(cfg, CMP_DATASET, CMP_IMB_RATIO)}_{CMP_KIND}"
vis.compare(folders, ["Long-tailed", "Re-balanced", "Balanced"], tag, out_dir, name)
display(Image(os.path.join(out_dir, f"{name}_{tag}.png"), width=950))

In [ ]:
#@title 4.5 Download all tables and figures as one zip (no checkpoints)
import zipfile
zip_path = "/content/resa_results_tables_and_figures.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for dirpath, _, filenames in os.walk(RESULTS_ROOT):
        for fn in filenames:
            if fn.endswith((".csv", ".png", ".json", ".npy")):
                full = os.path.join(dirpath, fn)
                z.write(full, os.path.relpath(full, RESULTS_ROOT))
print(zip_path, f"{os.path.getsize(zip_path) / 1e6:.1f} MB")
from google.colab import files
files.download(zip_path)

## Appendix — smoke test (about two minutes on a GPU)

Runs the whole pipeline with tiny settings into a separate folder (`RESULTS_ROOT/_smoke_test`), to confirm that the code, the paths and the datasets work before a long run is started. The numbers it prints are meaningless.

In [ ]:
#@title Smoke test: 2 pretraining epochs, then every evaluation once
import shutil
smoke = ex.config(results_root=os.path.join(RESULTS_ROOT, "_smoke_test"), data_root=DATA_ROOT,
                  pretrain_epochs=2, probe_epochs=2, probe_optimizer=PROBE_OPTIMIZER,
                  probe_lr=PROBE_LR, probe_lr_end=PROBE_LR_END, probe_wd=PROBE_WD,
                  probe_batch_size=PROBE_BATCH_SIZE, workers=WORKERS)
shutil.rmtree(smoke["results_root"], ignore_errors=True)
ex.pretrain(smoke, "cifar10", 100)
ex.knn(smoke, "cifar10", 100, "rebalanced", [5, 10, 20])
ex.linear_probe(smoke, "cifar10", 100, "rebalanced")
ex.transfer(smoke, "cifar10", 100, "cifar100", [5, 10, 20])
import collect_results, visualise_results as vis
collect_results.collect(smoke["results_root"])
vis.visualise(os.path.join(ex.eval_dir(smoke, "cifar10", 100, "knn", "rebalanced"), "metrics"), "k10")
print("\nSmoke test passed. The folder", smoke["results_root"], "can be deleted.")